Лабораторная работа 4 Улучшение качества модели

Цель: Научиться повышать качество моделей и сравнивать алгоритмы машинного обучения.

Предметная область: Анализ авиаперевозок и факторов, влияющих на задержки рейсов.

Задача: Улучшить качество модели предсказания задержек рейсов.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [3]:
df = pd.read_csv("finish_laba3.csv")
df.head()

,year,month,carrier,airport,arr_flights,arr_del15,carrier_ct,weather_ct,nas_ct,security_ct,...,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,delay_per_flight,total_delay
0,2022,7,9E,ABE,33.0,2.0,0.92,1.00,0.08,0.0,...,0.0,0.0,129.0,98.0,23.0,8.0,0.0,0.0,3.794118,129.0
1,2022,7,9E,ABY,78.0,25.0,11.80,0.72,5.01,0.0,...,0.0,0.0,1664.0,887.0,52.0,224.0,0.0,501.0,21.063291,1664.0
2,2022,7,9E,ACK,124.0,19.0,5.84,1.00,6.76,0.0,...,5.0,4.0,1523.0,388.0,35.0,511.0,0.0,589.0,12.184000,1523.0
3,2022,7,9E,AEX,67.0,10.0,1.32,1.00,2.40,1.0,...,0.0,1.0,657.0,103.0,82.0,93.0,25.0,354.0,9.661765,632.0
4,2022,7,9E,AGS,174.0,30.0,18.10,5.75,3.60,0.0,...,1.0,0.0,2462.0,1686.0,310.0,139.0,0.0,327.0,14.068571,2462.0


In [8]:
X = df.drop("arr_delay", axis=1)
X = X.select_dtypes(include=['number'])
y = df["arr_delay"]

Выполняется разделение данных на признаки и целевую переменную

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

Разделение на обучающую и тестовую выборки

In [10]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

Базовая модель

In [11]:
print("MAE:", mean_absolute_error(y_test, y_pred_lr))
print("MSE:", mean_squared_error(y_test, y_pred_lr))
print("R2:", r2_score(y_test, y_pred_lr))

MAE: 0.17099827943591484
MSE: 0.3366528825914125
R2: 0.9999997948358543


Оценка качества
Обучается базовая линейная модель
Оценивается качество по MAE, MSE, R2

In [12]:
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)

Альтернативная модель Decision Tree

In [13]:
param_grid = {
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5, 10]
}

grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring="r2"
)

grid.fit(X_train, y_train)

best_tree = grid.best_estimator_

Подбор гиперпараметров

In [14]:
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

Random Forest (улучшенная модель)

In [15]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Decision Tree", "Random Forest"],
    "MAE": [
        mean_absolute_error(y_test, y_pred_lr),
        mean_absolute_error(y_test, y_pred_tree),
        mean_absolute_error(y_test, y_pred_rf)
    ],
    "MSE": [
        mean_squared_error(y_test, y_pred_lr),
        mean_squared_error(y_test, y_pred_tree),
        mean_squared_error(y_test, y_pred_rf)
    ],
    "R2": [
        r2_score(y_test, y_pred_lr),
        r2_score(y_test, y_pred_tree),
        r2_score(y_test, y_pred_rf)
    ]
})

results

,Model,MAE,MSE,R2
0,Linear Regression,0.170998,0.336653,1.000000
1,Decision Tree,1.083601,80.317560,0.999951
2,Random Forest,0.725154,63.890783,0.999961


Сравнение моделей

In [16]:
print("Train R2 (Tree):", best_tree.score(X_train, y_train))
print("Test R2 (Tree):", best_tree.score(X_test, y_test))

Train R2 (Tree): 0.9999944136188432
Test R2 (Tree): 0.9999536946901698


Анализ переобучения

Вывод:
Обучение нескольких моделей
Подбор гиперпараметров
Сравнение качества моделей
Анализ переобучения